In [ ]:
import pandas as pd
import numpy as np

import sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [3]:
df_aps = pd.read_csv("https://minio.lab.sspcloud.fr/juleslapeuf/df_aps.csv")

In [4]:
df_aps.head()

,event_id,tournament,uniquetournament_id,season,season_id,round,bestOfSets,bestOfLegs,home_score,away_score,...,HighestCheckout_away,CheckoutsOver100_home,CheckoutsOver100_away,CheckoutsAccuracy_home,CheckoutsAccuracy_away,CheckoutsAccuracy_homeTotal,CheckoutsAccuracy_awayTotal,date,R_home_new,R_away_new
0,15142767,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,1.0,3.0,...,62.0,1.0,0.0,5.0,10.0,13.0,25.0,2025-12-11 19:15:00,0.326050,0.002538
1,15142749,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,0.0,...,72.0,0.0,0.0,9.0,3.0,25.0,10.0,2025-12-11 20:12:00,0.857104,-0.049318
2,15142718,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,0.0,...,130.0,2.0,1.0,9.0,5.0,14.0,11.0,2025-12-11 21:04:00,1.719075,-0.058423
3,15142764,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,1.0,...,92.0,0.0,0.0,11.0,8.0,35.0,22.0,2025-12-11 21:59:00,0.107832,-0.138110
4,15142751,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,0.0,...,100.0,1.0,1.0,9.0,4.0,25.0,14.0,2025-12-12 12:49:00,0.206039,-0.189061


In [5]:
df_aps.columns

Index(['event_id', 'tournament', 'uniquetournament_id', 'season', 'season_id',
       'round', 'bestOfSets', 'bestOfLegs', 'home_score', 'away_score',
       'home_id', 'away_id', 'home_name', 'away_name', 'Average3Darts_home',
       'Average3Darts_away', 'Thrown180_home', 'Thrown180_away',
       'ThrownOver140_home', 'ThrownOver140_away', 'ThrownOver100_home',
       'ThrownOver100_away', 'HighestCheckout_home', 'HighestCheckout_away',
       'CheckoutsOver100_home', 'CheckoutsOver100_away',
       'CheckoutsAccuracy_home', 'CheckoutsAccuracy_away',
       'CheckoutsAccuracy_homeTotal', 'CheckoutsAccuracy_awayTotal', 'date',
       'R_home_new', 'R_away_new'],
      dtype='object')

Ce jeu de données est un ensemble de matchs de fléchettes (un ligne = un match) :
- event_id : identifiant unique de chaque match
- tournamnent : nom du tournoi dans lequel le match s'est joué (avec uniquetournament_id son code)
- season : la saison du tournoi (season_id étant l'identifiant)
- round : le niveau du match dans le tournoi (finale, demi-finale...)
- bestOfSets : le match s'est joué au meilleur des ... sets
- bestOfLegs : les sets se sont joués au meilleur des ... manches
- home_score/away_score : nb de sets de chaque joueurs
- home_id/away_id / home_name/away_name : identifiant unique et et nom des joueurs
- date : la date exacte (à la minute) du match

Les autres colonnes correspondent aux statistiques du matchs. Pour donner un peu de contexte, pour une manche, les deux joueurs partent avec un total de 501 et ils enlèvent de leur total la somme des points des 3 fléchettes et cela ainsi de suite jusqu'à soustraire exactement 501 au total. Le premier qui soustrait 501 remporte la manche.
Le maximum de points possibles pour un lancer de trois fléchettes est 180 points, c'est-à-dire les 3 fléchettes dans la case triple 20. Il est impossible de terminer par une fléchette triple, donc le plus haut checkout (c'est-à-dire fin de manche) est 170, à savoir deux triple 20 et le rond du milieu (bulleseye) pour 50. Bref, c'est du détail, mais ça permet de comprendre un peu les ordres de grandeur.

- Average3Darts : La moyenne des 3 fléchettes sur l'ensemble du match
- Thrown180 : Nombre de lancers à 180
- ThrownOver140/Over100 : Nombre de lancers au-delà de 140, de 100
- HighestCheckout : La fin de manche la plus élevée (max 170)
- CheckoutsOver100 : Nombre de fin de manche au-dessus de 100
- CheckoutsAccuracy : Réussite des fins de mancher (nb de fois où le joueur a terminé la manche quand il le pouvait)

Enfin, j'ai calculé à partir d'un score de type ELO (connu aux échecs notamment) un rating pour chaque joueur qui évolue dans le temps

- R_home_new : rating du joueur home
- R_away_new : rating du joueur away

In [6]:
# Nombre d'apparitions de chaque joueur
player_counts = pd.concat([
    df_aps["home_id"],
    df_aps["away_id"]
]).value_counts()

# Joueurs apparaissant au moins 4 fois
eligible_players = player_counts[player_counts >= 4].index

# Conservation des matchs entre joueurs éligibles
df_aps_filtered = df_aps[
    df_aps["home_id"].isin(eligible_players)
    & df_aps["away_id"].isin(eligible_players)
].reset_index(drop=True).copy()

In [7]:
len(df_aps_filtered)

3682

In [8]:
for tournament in df_aps_filtered["tournament"].dropna().unique():
    print(tournament)

PDC World Championship 2026
PDC World Championship 2025
PDC World Championship 2024
PDC World Championship 2022/23
European Championship 2025
European Championship 2024
European Championship 2023
European Championship 2022
Grand Slam of Darts 2025, Knockout Stage
Grand Slam of Darts 2024, Knockout Stage
Grand Slam of Darts 2023, Knockout Stage
Grand Slam of Darts 2022, Knockout Stage
World Series of Darts Finals 2025
World Series of Darts Finals 2024
World Series of Darts Finals 2023
World Series of Darts Finals 2022
New Zealand Darts Masters 2025
New Zealand Darts Masters 2024
New Zealand Darts Masters 2023
New Zealand Darts Masters 2022
US Darts Masters 2025
US Darts Masters 2024
US Darts Masters 2023
European Tour 5 - Austrian Darts Open 2025
European Tour 5 - Austrian Darts Open 2024
European Tour 5 - Austrian Darts Open 2023
European Tour 11 - Czech Darts Open 2025
European Tour 13 - Czech Darts Open 2024
European Tour 8 - Czech Darts Open 2023
European Tour 6 - European Darts Gra

Le format set est utilisé par : PDC World Championship, World Grand Prix, World Masters.

In [9]:
tournois_en_sets = [
    "PDC World Championship 2026",
    "PDC World Championship 2025",
    "PDC World Championship 2024",
    "PDC World Championship 2022/23",
    "World Grand Prix 2025",
    "World Grand Prix 2024",
    "World Grand Prix 2023",
    "World Grand Prix 2022",
    "World Grand Prix",
    "World Masters 2026",
    "World Masters 2025"
]

df_aps_filtered["legs_played"] = (
    df_aps_filtered["home_score"] + df_aps_filtered["away_score"]
) * df_aps_filtered["tournament"].isin(tournois_en_sets).map({True: 4, False: 1})

In [10]:
df_aps_filtered["CheckoutsAccuracy_prop_home"] = df_aps_filtered["CheckoutsAccuracy_home"] / df_aps_filtered["CheckoutsAccuracy_homeTotal"]
df_aps_filtered["CheckoutsAccuracy_prop_away"] = df_aps_filtered["CheckoutsAccuracy_away"] / df_aps_filtered["CheckoutsAccuracy_awayTotal"]

In [11]:
stats_a_normaliser = [
    "Thrown180",
    "ThrownOver140",
    "ThrownOver100",
    "CheckoutsOver100"
]

for stat in stats_a_normaliser:
    for joueur in ["home", "away"]:
        colonne = f"{stat}_{joueur}"
        df_aps_filtered[f"{colonne}_per_leg"] = (
        df_aps_filtered[colonne] / df_aps_filtered["legs_played"]
        )

In [12]:
df_aps_filtered.head()

,event_id,tournament,uniquetournament_id,season,season_id,round,bestOfSets,bestOfLegs,home_score,away_score,...,CheckoutsAccuracy_prop_home,CheckoutsAccuracy_prop_away,Thrown180_home_per_leg,Thrown180_away_per_leg,ThrownOver140_home_per_leg,ThrownOver140_away_per_leg,ThrownOver100_home_per_leg,ThrownOver100_away_per_leg,CheckoutsOver100_home_per_leg,CheckoutsOver100_away_per_leg
0,15142767,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,1.0,3.0,...,0.384615,0.400000,0.062500,0.1875,0.750000,0.875000,0.875000,1.375000,0.062500,0.000000
1,15142718,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,0.0,...,0.642857,0.454545,0.583333,0.2500,0.666667,0.916667,1.750000,0.916667,0.166667,0.083333
2,15142764,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,1.0,...,0.314286,0.363636,0.062500,0.2500,0.437500,0.500000,1.500000,1.125000,0.000000,0.000000
3,15142751,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,0.0,...,0.360000,0.285714,0.250000,0.2500,0.916667,0.583333,1.083333,1.416667,0.083333,0.083333
4,15142722,PDC World Championship 2026,616,PDC World Championship 2026,85121,64,5.0,5.0,3.0,2.0,...,0.295455,0.205882,0.200000,0.3000,0.650000,0.400000,0.950000,0.900000,0.000000,0.000000


In [36]:
df_final = df_aps_filtered[(df_aps_filtered["Average3Darts_home"] > 0) & (df_aps_filtered["Average3Darts_away"] > 0)].copy()

In [37]:
len(df_final)

2684

# PREPARATION FINALE DU DATASET

In [39]:
# Copie de sécurité du dataset original
matches = df_final.copy()

# Conversion de la date et tri chronologique
matches["date"] = pd.to_datetime(matches["date"])

matches = matches.sort_values(
    by=["date", "event_id"]
).reset_index(drop=True)

In [40]:
matches.head()

,event_id,tournament,uniquetournament_id,season,season_id,round,bestOfSets,bestOfLegs,home_score,away_score,...,CheckoutsAccuracy_prop_home,CheckoutsAccuracy_prop_away,Thrown180_home_per_leg,Thrown180_away_per_leg,ThrownOver140_home_per_leg,ThrownOver140_away_per_leg,ThrownOver100_home_per_leg,ThrownOver100_away_per_leg,CheckoutsOver100_home_per_leg,CheckoutsOver100_away_per_leg
0,12118884,UK Open 2024,670,UK Open 2024,58727,32,NaN,NaN,10.0,8.0,...,0.333333,0.380952,0.333333,0.388889,0.888889,0.555556,1.222222,1.111111,0.111111,0.000000
1,12118885,UK Open 2024,670,UK Open 2024,58727,32,NaN,NaN,10.0,7.0,...,0.400000,0.280000,0.411765,0.294118,0.764706,0.705882,1.000000,0.941176,0.058824,0.058824
2,12118883,UK Open 2024,670,UK Open 2024,58727,32,NaN,NaN,7.0,10.0,...,0.304348,0.416667,0.235294,0.176471,0.647059,0.352941,1.058824,1.588235,0.058824,0.058824
3,12118880,UK Open 2024,670,UK Open 2024,58727,32,NaN,NaN,8.0,10.0,...,0.421053,0.526316,0.277778,0.166667,0.333333,0.666667,1.055556,1.277778,0.111111,0.000000
4,12118881,UK Open 2024,670,UK Open 2024,58727,32,NaN,NaN,7.0,10.0,...,0.437500,0.625000,0.176471,0.058824,0.764706,0.705882,1.529412,1.294118,0.176471,0.235294


In [41]:
stats = [
    "Average3Darts",
    "Thrown180_per_leg",
    "ThrownOver140_per_leg",
    "ThrownOver100_per_leg",
    "HighestCheckout",
    "CheckoutsOver100_per_leg",
    "CheckoutsAccuracy",
]

In [42]:
# Colonnes du joueur home
home_columns = {
    "home_id": "player_id",
    "home_name": "player_name",
    "Average3Darts_home": "Average3Darts",
    "Thrown180_home_per_leg": "Thrown180_per_leg",
    "ThrownOver140_home_per_leg": "ThrownOver140_per_leg",
    "ThrownOver100_home_per_leg": "ThrownOver100_per_leg",
    "HighestCheckout_home": "HighestCheckout",
    "CheckoutsOver100_home_per_leg": "CheckoutsOver100_per_leg",
    "CheckoutsAccuracy_prop_home": "CheckoutsAccuracy"
}

players_home = (
    matches[
        ["event_id", "date"] + list(home_columns.keys())
    ]
    .rename(columns=home_columns)
    .assign(side="home")
)


# Colonnes du joueur away
away_columns = {
    "away_id": "player_id",
    "away_name": "player_name",
    "Average3Darts_away": "Average3Darts",
    "Thrown180_away_per_leg": "Thrown180_per_leg",
    "ThrownOver140_away_per_leg": "ThrownOver140_per_leg",
    "ThrownOver100_away_per_leg": "ThrownOver100_per_leg",
    "HighestCheckout_away": "HighestCheckout",
    "CheckoutsOver100_away_per_leg": "CheckoutsOver100_per_leg",
    "CheckoutsAccuracy_prop_away": "CheckoutsAccuracy"
}

players_away = (
    matches[
        ["event_id", "date"] + list(away_columns.keys())
    ]
    .rename(columns=away_columns)
    .assign(side="away")
)


# Une ligne par joueur et par match
players_history = pd.concat(
    [players_home, players_away],
    ignore_index=True
).sort_values(
    by=["player_id", "date", "event_id"]
).reset_index(drop=True)

In [43]:
for stat in stats:
    players_history[f"{stat}_last3"] = (
        players_history
        .groupby("player_id")[stat]
        .transform(
            lambda serie: (
                serie
                .shift(1)
                .rolling(window=3, min_periods=3)
                .mean()
            )
        )
    )

In [44]:
history_columns = [
    "event_id",
    "player_id"
] + [f"{stat}_last3" for stat in stats]

history_home = (
    players_history[history_columns]
    .rename(columns={"player_id": "home_id"})
    .rename(
        columns={
            f"{stat}_last3": f"home_{stat}_last3"
            for stat in stats
        }
    )
)

history_away = (
    players_history[history_columns]
    .rename(columns={"player_id": "away_id"})
    .rename(
        columns={
            f"{stat}_last3": f"away_{stat}_last3"
            for stat in stats
        }
    )
)

In [45]:
dataset_ml = (
    matches
    .merge(
        history_home,
        on=["event_id", "home_id"],
        how="left"
    )
    .merge(
        history_away,
        on=["event_id", "away_id"],
        how="left"
    )
)

In [46]:
# 1 si le joueur home gagne, 0 sinon
dataset_ml["home_win"] = (
    dataset_ml["home_score"] > dataset_ml["away_score"]
).astype(int)

In [47]:
historical_features = (
    [f"home_{stat}_last3" for stat in stats]
    + [f"away_{stat}_last3" for stat in stats]
)

dataset_ml = dataset_ml.dropna(
    subset=historical_features
).reset_index(drop=True)

In [48]:
columns_finales = [
    "event_id",
    "date",
    "tournament",
    "season",
    "round",
    "home_id",
    "home_name",
    "away_id",
    "away_name",

    # Ratings déjà calculés
    "R_home_new",
    "R_away_new",

    # Variable cible
    "home_win"
] + historical_features

dataset_ml = dataset_ml[columns_finales]

In [49]:
print(dataset_ml.shape)
display(dataset_ml.head())

(1520, 26)


,event_id,date,tournament,season,round,home_id,home_name,away_id,away_name,R_home_new,...,home_HighestCheckout_last3,home_CheckoutsOver100_per_leg_last3,home_CheckoutsAccuracy_last3,away_Average3Darts_last3,away_Thrown180_per_leg_last3,away_ThrownOver140_per_leg_last3,away_ThrownOver100_per_leg_last3,away_HighestCheckout_last3,away_CheckoutsOver100_per_leg_last3,away_CheckoutsAccuracy_last3
0,12121796,2024-03-03 14:50:00,UK Open 2024,UK Open 2024,27,172078,Damon Heta,412122,Luke Littler,0.413643,...,109.666667,0.071150,0.432229,101.666667,0.141830,0.620479,1.294336,138.666667,0.133987,0.563142
1,12124623,2024-03-03 21:15:00,UK Open 2024,UK Open 2024,29,136790,Dimitri Van den Bergh,272992,Luke Humphries,0.612455,...,123.666667,0.061438,0.461538,99.666667,0.311050,0.912393,1.136752,107.666667,0.051587,0.453391
2,12205571,2024-04-01 17:00:00,European Tour 2 - German Darts Grand Prix 2024,PDC European Tour - German Darts Grand Prix 2024,27,407830,Josh Rock,143420,Gerwyn Price,0.167694,...,121.666667,0.111255,0.412179,103.990000,0.381287,0.923392,1.256140,67.666667,0.000000,0.372727
3,12205574,2024-04-01 17:30:00,European Tour 2 - German Darts Grand Prix 2024,PDC European Tour - German Darts Grand Prix 2024,27,222546,Danny Noppert,272992,Luke Humphries,0.463959,...,127.666667,0.067340,0.511111,105.436667,0.382275,1.016534,1.057540,122.333333,0.126984,0.548774
4,12205575,2024-04-01 17:55:00,European Tour 2 - German Darts Grand Prix 2024,PDC European Tour - German Darts Grand Prix 2024,27,190416,Nathan Aspinall,163422,Martin Schindler,0.589926,...,115.333333,0.035088,0.370980,98.433333,0.273016,0.707937,1.216931,107.333333,0.080952,0.505430


# PRemier modèle RL

In [50]:
# Variables historiques des joueurs
historical_features = (
    [f"home_{stat}_last3" for stat in stats]
    + [f"away_{stat}_last3" for stat in stats]
)

# Ajout des ratings existants
features = [
    "R_home_new",
    "R_away_new"
] + historical_features

target = "home_win"

# Copie et tri chronologique
model_data = (
    dataset_ml
    .sort_values("date")
    .reset_index(drop=True)
)

X = model_data[features]
y = model_data[target]

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    shuffle=False
)

print("Observations d'entraînement :", len(X_train))
print("Observations de test :", len(X_test))

Observations d'entraînement : 1216
Observations de test : 304


In [52]:
logistic_model = Pipeline(
    steps=[
        # Remplacement des éventuelles valeurs manquantes
        ("imputer", SimpleImputer(strategy="median")),

        # Mise à l'échelle des variables
        ("scaler", StandardScaler()),

        # Régression logistique
        ("logistic_regression", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

logistic_model.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('logistic_regression',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [53]:
# Classes prédites : 0 ou 1
y_pred = logistic_model.predict(X_test)

# Probabilité que le joueur home gagne
y_proba_home = logistic_model.predict_proba(X_test)[:, 1]

In [54]:
accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba_home)

print(f"Accuracy : {accuracy:.3f}")
print(f"Balanced accuracy : {balanced_accuracy:.3f}")
print(f"ROC AUC : {roc_auc:.3f}")

print("\nRapport de classification :")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Victoire away", "Victoire home"]
    )
)

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.628
Balanced accuracy : 0.600
ROC AUC : 0.602

Rapport de classification :
               precision    recall  f1-score   support

Victoire away       0.50      0.49      0.49       112
Victoire home       0.70      0.71      0.71       192

     accuracy                           0.63       304
    macro avg       0.60      0.60      0.60       304
 weighted avg       0.63      0.63      0.63       304

Matrice de confusion :
[[ 55  57]
 [ 56 136]]


In [55]:
results_test = model_data.iloc[X_test.index].copy()

results_test["prediction_home_win"] = y_pred
results_test["probability_home_win"] = y_proba_home

results_test[
    [
        "event_id",
        "date",
        "home_name",
        "away_name",
        "home_win",
        "prediction_home_win",
        "probability_home_win"
    ]
].head()

,event_id,date,home_name,away_name,home_win,prediction_home_win,probability_home_win
1216,16477757,2026-07-10 18:43:00,Rob Cross,Max Hopp,0,1,0.817301
1217,16477760,2026-07-10 19:14:00,Joe Cullen,Charlie Manby,1,1,0.614769
1218,16477759,2026-07-10 19:38:00,Kevin Doets,Dominik Grüllich,1,1,0.705009
1219,16477761,2026-07-10 20:10:00,Niko Springer,Darius Labanauskas,1,1,0.584150
1220,16477762,2026-07-10 20:37:00,Niels Zonneveld,Klose D.,1,1,0.726955


In [56]:
coefficients = (
    logistic_model
    .named_steps["logistic_regression"]
    .coef_[0]
)

coefficients_df = pd.DataFrame({
    "feature": features,
    "coefficient": coefficients,
    "odds_ratio": np.exp(coefficients)
}).sort_values(
    by="coefficient",
    ascending=False
)

display(coefficients_df)

,feature,coefficient,odds_ratio
0,R_home_new,0.471255,1.602003
2,home_Average3Darts_last3,0.453399,1.573652
15,away_CheckoutsAccuracy_last3,0.133223,1.142505
12,away_ThrownOver100_per_leg_last3,0.103283,1.108805
14,away_CheckoutsOver100_per_leg_last3,0.033515,1.034083
6,home_HighestCheckout_last3,0.006458,1.006479
10,away_Thrown180_per_leg_last3,-0.006167,0.993852
13,away_HighestCheckout_last3,-0.015386,0.984732
11,away_ThrownOver140_per_leg_last3,-0.039551,0.961221
5,home_ThrownOver100_per_leg_last3,-0.065777,0.936340


In [57]:
results_test["probability_away_win"] = (
    1 - results_test["probability_home_win"]
)